# YOLO26s → DenseNet121 EXP-024: cascade inference

**EXP-025** cung cấp phân tích tiếp theo cho checkpoint EXP-024; không có model 025 riêng để nạp.

**Pipeline:** YOLO26s EXP-Y26S-SMALL-008 khoanh vùng lesion; DenseNet121 EXP-DENSENET121-TRANSFER-5SEEDS-024-256px-f9fd5d5f (deployment seed 42) phân loại từng crop. Notebook chỉ chạy **inference và đánh giá thăm dò**, không huấn luyện lại hai mô hình.

**Nguồn đã khóa:** [YOLO dataset V6](https://drive.google.com/file/d/1WGwaWJfFNFK8a3Po6GdjHuxFizXfYvwI/view), [DenseNet 024 model](https://drive.google.com/file/d/1DTBO6227tdsF29HCoXpfcJ4xAkSunB5t/view), [DenseNet 024 experiment (EXP-025 is later evaluation, not a new trained model)](https://drive.google.com/drive/folders/13BEQoXLIdhanQ1zSLRawhAeoBwAQji-M). DenseNet 024 được train trên **ảnh lá nguyên 256×256**, còn đầu vào ở đây là crop từ box YOLO. Đây là dịch chuyển miền; không coi độ tin cậy DenseNet là xác suất đã hiệu chỉnh cho crop. Test chỉ dùng để báo cáo, không chỉnh ngưỡng hay quy tắc dựa trên kết quả Test.

**Đầu ra:** CSV từng box, chỉ số detection YOLO và cascade trên cùng Test split, hình minh họa, config/hash nguồn; ZIP kết quả được upload vào thư mục YOLO trên Google Drive. YOLO chỉ có ba class bệnh; DenseNet có thêm `healthy` và `not_cashew_leaf`. Hai class ngoài ba bệnh được xử lý như từ chối box trong cascade.


## 01. Chạy trên Colab hoặc Kaggle

**Colab:** chạy từ đầu notebook, cấp quyền Google khi cell xác thực hiện hộp thoại. Notebook dùng Google Drive API để đọc model/dataset và upload ZIP kết quả; không cần `kaggle_secrets` hay mount Drive. File tạm nằm trong `/content`.

**Kaggle:** bật Internet và gắn secret `GOOGLE_DRIVE_OAUTH_JSON` như notebook train YOLO26. File tạm nằm trong `/kaggle/working`.

Notebook tự tìm YOLO 008 `best.pt` trong phiên làm việc hiện tại, ZIP kết quả trong thư mục input, hoặc thư mục YOLO trên Drive. Nếu ZIP kết quả 008 chưa có trên Drive, hãy chạy cell upload ở notebook YOLO trước hoặc đặt đường dẫn ZIP vào `YOLO_RESULT_ZIP_OVERRIDE`. Không dùng checkpoint khác chỉ vì trùng tên `best.pt`.


In [1]:
!pip install -q -U ultralytics tensorflow==2.21.0 google-api-python-client google-auth pyyaml


In [2]:
import csv
import hashlib
import io
import json
import math
import shutil
import sys
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageDraw
import tensorflow as tf
from tensorflow import keras
import torch
import ultralytics
from ultralytics import YOLO
import google.auth
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

try:
    import google.colab  # Available only inside Google Colab.
    RUNTIME = "Colab"
except ImportError:
    RUNTIME = "Kaggle" if Path("/kaggle/working").is_dir() else "Unknown"
if RUNTIME == "Unknown":
    raise RuntimeError("Notebook này hỗ trợ Google Colab hoặc Kaggle")

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        print("TensorFlow GPU already initialized; keeping its current memory setting")

print("Runtime:", RUNTIME)
print("Python:", sys.version.split()[0], "TensorFlow:", tf.__version__,
      "Ultralytics:", ultralytics.__version__, "Torch:", torch.__version__)


Runtime: Colab
Python: 3.13.16 TensorFlow: 2.21.0 Ultralytics: 8.4.192 Torch: 2.11.0+cu130


In [3]:
# ============================================================
# 02. CONFIG — pinned source models and dataset
# ============================================================
EXPERIMENT_ID = "EXP-YOLO26S-DENSENET121-024-CASCADE-001"
YOLO_EXPERIMENT_ID = "EXP-Y26S-SMALL-008"
DENSENET_EXPERIMENT_ID = "EXP-DENSENET121-TRANSFER-5SEEDS-024-256px-f9fd5d5f"

YOLO_FOLDER_ID = "16S4O-7f7Tl2_xb0JrB5b-PJZA0FqKQ1M"
YOLO_DATASET_FILE_ID = "1WGwaWJfFNFK8a3Po6GdjHuxFizXfYvwI"
YOLO_DATASET_NAME = "Test_400image_V6.v1i.yolo26.zip"
YOLO_RESULT_ZIP_NAME = "EXP-Y26S-SMALL-008.zip"
YOLO_RESULT_ZIP_FILE_ID = None  # Optional exact ID if several same-name ZIPs exist.
YOLO_BEST_PT_OVERRIDE = None    # Optional exact local checkpoint path.
YOLO_RESULT_ZIP_OVERRIDE = None # Optional local ZIP path (e.g. uploaded to Colab).
YOLO_DATASET_ZIP_OVERRIDE = None # Optional local dataset ZIP path.

DENSENET_MODEL_ID = "1DTBO6227tdsF29HCoXpfcJ4xAkSunB5t"
DENSENET_MODEL_SHA256 = "8bcd23bcadfa83c81264feceba89eef40d9b5518b9497d4e51c50fe6a6d8dd0a"
DENSENET_LABEL_MAP_ID = "1Vz6PSjAHE2CHUEiR3Y8BcrOpH0Jfo4R0"
DENSENET_LABEL_MAP_SHA256 = "f426a64d2c133785b398dd2c0afd4e33d2a8db5de098e4e6e008b9a1b97bc995"
DENSENET_METADATA_ID = "15QJUU3Dmb7rx4b6PySpJmuB3iJ0_8wuL"
DENSENET_METADATA_SHA256 = "3544fa9f39d4b44f928c98e0b7aea8b555553ee250a30b0aa5aff2a8e5600d04"
DENSENET_DEPLOYMENT_SEED = 42

DISEASE_CLASSES = ["anthracnose", "leaf_miner", "red_rust"]
DENSENET_CLASSES = ["anthracnose", "healthy", "leaf_miner", "red_rust", "not_cashew_leaf"]
DETECT_CONF = 0.25  # Fixed before Test inspection.
NMS_IOU = 0.70
MATCH_IOU = 0.50
YOLO_IMG_SIZE = 640
DENSENET_IMG_SIZE = 256
DEVICE = 0 if torch.cuda.is_available() else "cpu"

WORK_BASE = Path("/content") if RUNTIME == "Colab" else Path("/kaggle/working")
LOCAL_INPUT_DIR = Path("/content/input") if RUNTIME == "Colab" else Path("/kaggle/input")
WORK_ROOT = WORK_BASE / EXPERIMENT_ID
ASSET_DIR = WORK_ROOT / "assets"
RESULT_DIR = WORK_ROOT / "results"
SAMPLE_DIR = RESULT_DIR / "samples"
for folder in (ASSET_DIR, RESULT_DIR, SAMPLE_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Experiment:", EXPERIMENT_ID, "Device:", DEVICE)


Experiment: EXP-YOLO26S-DENSENET121-024-CASCADE-001 Device: 0


In [4]:
# ============================================================
# 03. DRIVE AUTH PREFLIGHT — Colab sign-in or Kaggle Secret
# ============================================================
try:
    if RUNTIME == "Colab":
        from google.colab import auth
        auth.authenticate_user()
        credentials, _ = google.auth.default()
    else:
        from kaggle_secrets import UserSecretsClient
        secret_json = UserSecretsClient().get_secret("GOOGLE_DRIVE_OAUTH_JSON")
        secret_info = json.loads(secret_json)
        if secret_info.get("type") != "authorized_user":
            raise ValueError("Kaggle Secret phải là authorized_user JSON")
        credentials = Credentials.from_authorized_user_info(
            secret_info, scopes=["https://www.googleapis.com/auth/drive"]
        )
        if not credentials.valid:
            credentials.refresh(Request())
    drive_service = build("drive", "v3", credentials=credentials, cache_discovery=False)
    target_folder = drive_service.files().get(
        fileId=YOLO_FOLDER_ID,
        fields="id,name,mimeType,capabilities(canAddChildren)",
        supportsAllDrives=True,
    ).execute()
except Exception as exc:
    raise RuntimeError(
        "Không kết nối được Google Drive. Trên Colab hãy cấp quyền cho đúng tài khoản; "
        "trên Kaggle hãy bật Internet và gắn Secret GOOGLE_DRIVE_OAUTH_JSON."
    ) from exc
if target_folder.get("mimeType") != "application/vnd.google-apps.folder":
    raise ValueError("YOLO_FOLDER_ID không phải folder")
if not target_folder.get("capabilities", {}).get("canAddChildren", False):
    raise PermissionError("Tài khoản Google không có quyền ghi vào folder YOLO")
print("Drive ready:", target_folder["name"])


Drive ready: YOLO


In [5]:
# ============================================================
# 04. FILE HELPERS / AUTHENTICATED DRIVE DOWNLOAD
# ============================================================
def file_digest(path, algorithm="sha256"):
    h = hashlib.new(algorithm)
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def drive_download(file_id, destination, expected_sha256=None):
    destination = Path(destination)
    metadata = drive_service.files().get(
        fileId=file_id, fields="id,name,size,md5Checksum", supportsAllDrives=True
    ).execute()
    if destination.exists() and int(metadata["size"]) == destination.stat().st_size:
        cached_md5_ok = (not metadata.get("md5Checksum") or
                         file_digest(destination, "md5") == metadata["md5Checksum"])
        cached_sha_ok = (expected_sha256 is None or
                         file_digest(destination) == expected_sha256)
        if cached_md5_ok and cached_sha_ok:
            return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp = destination.with_name(destination.name + ".partial")
    request = drive_service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with temp.open("wb") as stream:
        downloader = MediaIoBaseDownload(stream, request, chunksize=8 * 1024 * 1024)
        done = False
        while not done:
            status, done = downloader.next_chunk(num_retries=3)
            if status is not None:
                print(f"Downloading {metadata['name']}: {status.progress():.0%}")
    if temp.stat().st_size != int(metadata["size"]):
        raise IOError(f"Download size mismatch: {metadata['name']}")
    if metadata.get("md5Checksum") and file_digest(temp, "md5") != metadata["md5Checksum"]:
        raise IOError(f"Download MD5 mismatch: {metadata['name']}")
    if expected_sha256 and file_digest(temp) != expected_sha256:
        raise IOError(f"Source SHA256 mismatch: {metadata['name']}")
    temp.replace(destination)
    return destination

def safe_extract(archive_path, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f"Unsafe ZIP entry: {member.filename}")
        archive.extractall(destination)
    return destination


In [6]:
# ============================================================
# 05. LOAD EXACT YOLO 008 CHECKPOINT
# ============================================================
checkpoint_override = Path(YOLO_BEST_PT_OVERRIDE) if YOLO_BEST_PT_OVERRIDE else None
session_checkpoint = WORK_BASE / "cashew_yolo26" / "runs" / "train" / YOLO_EXPERIMENT_ID / "weights" / "best.pt"
if checkpoint_override is not None:
    YOLO_BEST_PT = checkpoint_override
elif session_checkpoint.is_file():
    YOLO_BEST_PT = session_checkpoint
else:
    if YOLO_RESULT_ZIP_OVERRIDE:
        yolo_archive = Path(YOLO_RESULT_ZIP_OVERRIDE)
        if not yolo_archive.is_file():
            raise FileNotFoundError(f"YOLO_RESULT_ZIP_OVERRIDE không tồn tại: {yolo_archive}")
    else:
        input_archives = (sorted(LOCAL_INPUT_DIR.rglob(YOLO_RESULT_ZIP_NAME))
                          if LOCAL_INPUT_DIR.is_dir() else [])
        if len(input_archives) > 1:
            raise RuntimeError(f"Nhiều ZIP YOLO 008 trong input: {input_archives}")
        if input_archives:
            yolo_archive = input_archives[0]
        else:
            if YOLO_RESULT_ZIP_FILE_ID:
                result_file_id = YOLO_RESULT_ZIP_FILE_ID
            else:
                result_files = drive_service.files().list(
                    q=f"'{YOLO_FOLDER_ID}' in parents and name = '{YOLO_RESULT_ZIP_NAME}' and trashed = false",
                    fields="files(id,name,size,md5Checksum)", pageSize=100,
                    supportsAllDrives=True, includeItemsFromAllDrives=True,
                ).execute().get("files", [])
                if len(result_files) != 1:
                    raise FileNotFoundError(
                        f"Cần đúng một {YOLO_RESULT_ZIP_NAME} trong folder YOLO, thấy {len(result_files)}. "
                        "Chạy cell upload notebook YOLO 008 hoặc điền YOLO_RESULT_ZIP_OVERRIDE."
                    )
                result_file_id = result_files[0]["id"]
            yolo_archive = drive_download(result_file_id, ASSET_DIR / YOLO_RESULT_ZIP_NAME)
    YOLO_BEST_PT = ASSET_DIR / f"{YOLO_EXPERIMENT_ID}_best.pt"
    with zipfile.ZipFile(yolo_archive) as archive:
        matches = [name for name in archive.namelist()
                   if name.endswith(f"{YOLO_EXPERIMENT_ID}_best.pt")]
        if len(matches) != 1:
            raise RuntimeError(f"ZIP kết quả YOLO thiếu checkpoint 008 duy nhất: {matches}")
        with archive.open(matches[0]) as source, YOLO_BEST_PT.open("wb") as target:
            shutil.copyfileobj(source, target)
if not YOLO_BEST_PT.is_file() or YOLO_BEST_PT.stat().st_size == 0:
    raise FileNotFoundError(f"YOLO best.pt không hợp lệ: {YOLO_BEST_PT}")
YOLO_BEST_SHA256 = file_digest(YOLO_BEST_PT)
print("YOLO checkpoint:", YOLO_BEST_PT, "SHA256:", YOLO_BEST_SHA256)


FileNotFoundError: Cần đúng một EXP-Y26S-SMALL-008.zip trong folder YOLO, thấy 0. Chạy cell upload notebook YOLO 008 hoặc điền YOLO_RESULT_ZIP_OVERRIDE.

In [ ]:
# ============================================================
# 06. LOAD PINNED DENSENET 026 DEPLOYMENT PACKAGE
# ============================================================
dense_model_path = drive_download(
    DENSENET_MODEL_ID, ASSET_DIR / "densenet024_deployment_model.keras", DENSENET_MODEL_SHA256
)
label_map_path = drive_download(
    DENSENET_LABEL_MAP_ID, ASSET_DIR / "densenet024_label_map.json", DENSENET_LABEL_MAP_SHA256
)
metadata_path = drive_download(
    DENSENET_METADATA_ID, ASSET_DIR / "densenet024_inference_metadata.json", DENSENET_METADATA_SHA256
)
label_map = json.loads(label_map_path.read_text(encoding="utf-8"))
dense_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
if [label_map[str(i)] for i in range(5)] != DENSENET_CLASSES:
    raise ValueError("DenseNet 024 label_map không khớp class order đã khóa")
if dense_metadata.get("deployment_seed") != DENSENET_DEPLOYMENT_SEED:
    raise ValueError("DenseNet 024 deployment seed không phải 2026")
if dense_metadata.get("class_names") != DENSENET_CLASSES:
    raise ValueError("DenseNet 024 inference metadata không khớp")
if "256x256" not in dense_metadata.get("input", ""):
    raise ValueError("DenseNet 024 input size khác 224")

detector = YOLO(str(YOLO_BEST_PT))
yolo_names = [detector.names[i] for i in range(len(detector.names))]
if yolo_names != DISEASE_CLASSES:
    raise ValueError(f"YOLO 008 class order sai: {yolo_names}")
classifier = keras.models.load_model(dense_model_path, compile=False)
if tuple(classifier.input_shape[1:]) != (256, 256, 3):
    raise ValueError(f"DenseNet 024 input shape sai: {classifier.input_shape}")
if classifier.output_shape[-1] != 5:
    raise ValueError(f"DenseNet 024 output shape sai: {classifier.output_shape}")
smoke = classifier(np.zeros((1, 256, 256, 3), dtype=np.float32), training=False).numpy()
if smoke.shape != (1, 5) or not np.isfinite(smoke).all():
    raise RuntimeError("DenseNet 024 smoke inference thất bại")
print("Models loaded:", yolo_names, "->", DENSENET_CLASSES)


In [ ]:
# ============================================================
# 07. LOAD THE SAME YOLO V6 TEST SPLIT
# ============================================================
if YOLO_DATASET_ZIP_OVERRIDE:
    dataset_zip = Path(YOLO_DATASET_ZIP_OVERRIDE)
    if not dataset_zip.is_file():
        raise FileNotFoundError(f"YOLO_DATASET_ZIP_OVERRIDE không tồn tại: {dataset_zip}")
else:
    input_datasets = (sorted(LOCAL_INPUT_DIR.rglob(YOLO_DATASET_NAME))
                      if LOCAL_INPUT_DIR.is_dir() else [])
    if len(input_datasets) > 1:
        raise RuntimeError(f"Nhiều dataset ZIP cùng tên: {input_datasets}")
    dataset_zip = input_datasets[0] if input_datasets else drive_download(
        YOLO_DATASET_FILE_ID, ASSET_DIR / YOLO_DATASET_NAME
    )
DATASET_SHA256 = file_digest(dataset_zip)
data_dir = safe_extract(dataset_zip, WORK_ROOT / "dataset")
yaml_files = list(data_dir.rglob("data.yaml")) + list(data_dir.rglob("data.yml"))
if len(yaml_files) != 1:
    raise RuntimeError(f"Cần đúng một data.yaml/data.yml: {yaml_files}")
dataset_root = yaml_files[0].parent
data_cfg = yaml.safe_load(yaml_files[0].read_text(encoding="utf-8"))
if data_cfg.get("names") != DISEASE_CLASSES or int(data_cfg.get("nc", -1)) != 3:
    raise ValueError(f"YOLO dataset class map sai: {data_cfg.get('names')}")
test_image_dir = dataset_root / "test" / "images"
test_label_dir = dataset_root / "test" / "labels"
if not test_image_dir.is_dir() or not test_label_dir.is_dir():
    raise FileNotFoundError("Dataset ZIP thiếu test/images hoặc test/labels")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
test_images = sorted(p for p in test_image_dir.iterdir()
                     if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
if not test_images:
    raise RuntimeError("Test split không có ảnh")
print("Test images:", len(test_images), "Dataset SHA256:", DATASET_SHA256)


## 08. Inference theo box

Mỗi box YOLO được crop từ ảnh gốc, resize bilinear về 256×256 và đưa vào DenseNet dưới dạng RGB float32 0..255. `deployment_model.keras` đã chứa preprocessing; không chia 255 lần nữa. Giữ riêng confidence của YOLO và DenseNet; không nhân hai confidence vì chúng chưa được hiệu chỉnh chung.


In [ ]:
def load_gt_boxes(image_path, image_size):
    label_path = test_label_dir / f"{image_path.stem}.txt"
    if not label_path.exists():
        return []
    width, height = image_size
    gt = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        fields = line.split()
        if not fields:
            continue
        if len(fields) != 5:
            raise ValueError(f"Invalid YOLO label: {label_path}: {line}")
        class_id = int(fields[0])
        xc, yc, bw, bh = map(float, fields[1:])
        if class_id not in range(3) or not all(0 <= v <= 1 for v in (xc, yc, bw, bh)):
            raise ValueError(f"Invalid class/coordinates: {label_path}: {line}")
        gt.append({
            "class": DISEASE_CLASSES[class_id],
            "box": ((xc - bw / 2) * width, (yc - bh / 2) * height,
                    (xc + bw / 2) * width, (yc + bh / 2) * height),
        })
    return gt

def infer_image(image_path):
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")
    result = detector.predict(
        source=str(image_path), imgsz=YOLO_IMG_SIZE, conf=DETECT_CONF,
        iou=NMS_IOU, device=DEVICE, verbose=False,
    )[0]
    rows, crop_inputs = [], []
    if result.boxes is not None:
        coords = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)
        scores = result.boxes.conf.cpu().numpy()
        for detection_index, (box, class_id, score) in enumerate(zip(coords, classes, scores)):
            x1, y1, x2, y2 = map(float, box)
            left = max(0, min(image.width, math.floor(x1)))
            top = max(0, min(image.height, math.floor(y1)))
            right = max(0, min(image.width, math.ceil(x2)))
            bottom = max(0, min(image.height, math.ceil(y2)))
            if right <= left or bottom <= top:
                continue
            crop = image.crop((left, top, right, bottom))
            raw_rgb = np.asarray(crop, dtype=np.float32)
            resized = tf.image.resize(raw_rgb, (256, 256), method="bilinear").numpy()
            crop_inputs.append(resized)
            rows.append({
                "image": image_path.name, "detection_index": detection_index,
                "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                "yolo_class": DISEASE_CLASSES[int(class_id)],
                "yolo_conf": float(score),
            })
    if crop_inputs:
        probabilities = classifier.predict(np.stack(crop_inputs), batch_size=16, verbose=0)
        for row, probs in zip(rows, probabilities):
            pred_id = int(np.argmax(probs))
            row["densenet_class"] = DENSENET_CLASSES[pred_id]
            row["densenet_conf"] = float(probs[pred_id])
            row["agreement"] = row["yolo_class"] == row["densenet_class"]
            row["cascade_class"] = row["densenet_class"] if row["densenet_class"] in DISEASE_CLASSES else "rejected"
            for i, name in enumerate(DENSENET_CLASSES):
                row[f"dense_prob_{name}"] = float(probs[i])
    return rows, load_gt_boxes(image_path, image.size)

predictions_by_image, ground_truth_by_image = {}, {}
all_predictions = []
for index, image_path in enumerate(test_images, 1):
    rows, gt = infer_image(image_path)
    predictions_by_image[image_path.name] = rows
    ground_truth_by_image[image_path.name] = gt
    all_predictions.extend(rows)
    if index % 20 == 0 or index == len(test_images):
        print(f"Processed {index}/{len(test_images)} images")

prediction_columns = [
    "image", "detection_index", "x1", "y1", "x2", "y2", "yolo_class",
    "yolo_conf", "densenet_class", "densenet_conf", "agreement", "cascade_class",
] + [f"dense_prob_{name}" for name in DENSENET_CLASSES]
prediction_df = pd.DataFrame(all_predictions, columns=prediction_columns)
prediction_df.to_csv(RESULT_DIR / "box_predictions.csv", index=False)
display(prediction_df.head(10))


In [ ]:
# ============================================================
# 09. CLASS-AWARE BOX MATCHING ON THE FROZEN TEST SPLIT
# ============================================================
def box_iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def evaluate_predictions(class_field):
    counts = {name: {"tp": 0, "fp": 0, "fn": 0} for name in DISEASE_CLASSES}
    rejected_boxes = 0
    for image_path in test_images:
        name = image_path.name
        gt = ground_truth_by_image[name]
        unmatched = set(range(len(gt)))
        predictions = sorted(predictions_by_image[name],
                             key=lambda row: row["yolo_conf"], reverse=True)
        for pred in predictions:
            pred_class = pred[class_field]
            if pred_class not in counts:
                rejected_boxes += 1
                continue
            pred_box = (pred["x1"], pred["y1"], pred["x2"], pred["y2"])
            eligible = [i for i in unmatched if gt[i]["class"] == pred_class]
            best = max(eligible, key=lambda i: box_iou(pred_box, gt[i]["box"]), default=None)
            if best is not None and box_iou(pred_box, gt[best]["box"]) >= MATCH_IOU:
                counts[pred_class]["tp"] += 1
                unmatched.remove(best)
            else:
                counts[pred_class]["fp"] += 1
        for i in unmatched:
            counts[gt[i]["class"]]["fn"] += 1
    rows = []
    for class_name in DISEASE_CLASSES:
        item = counts[class_name]
        tp, fp, fn = item["tp"], item["fp"], item["fn"]
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({"class": class_name, **item,
                     "precision": precision, "recall": recall, "f1": f1})
    return rows, rejected_boxes

yolo_rows, _ = evaluate_predictions("yolo_class")
cascade_rows, rejected_boxes = evaluate_predictions("cascade_class")
metric_rows = ([{"model": "YOLO only", **row} for row in yolo_rows] +
               [{"model": "YOLO → DenseNet024", **row} for row in cascade_rows])
metric_df = pd.DataFrame(metric_rows)
metric_df.to_csv(RESULT_DIR / "test_box_metrics.csv", index=False)

def micro_summary(rows):
    tp, fp, fn = (sum(row[key] for row in rows) for key in ("tp", "fp", "fn"))
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": p, "recall": r,
            "f1": 2 * p * r / (p + r) if p + r else 0.0}

summary = {
    "yolo_only": micro_summary(yolo_rows),
    "cascade": micro_summary(cascade_rows),
    "rejected_boxes": rejected_boxes,
    "num_test_images": len(test_images),
    "num_yolo_boxes": len(all_predictions),
    "agreement_rate_on_yolo_boxes": (
        float(prediction_df["agreement"].mean()) if len(prediction_df) else None
    ),
    "evaluation_note": "Exploratory Test result. DenseNet024 trained on full leaves, not YOLO lesion crops. No Test tuning.",
}
(RESULT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(metric_df)
print(json.dumps(summary, indent=2))


In [ ]:
# ============================================================
# 10. SAMPLE ANNOTATED IMAGES
# ============================================================
for image_path in test_images[:8]:
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    for pred in predictions_by_image[image_path.name]:
        color = "#16a34a" if pred["agreement"] else "#f97316"
        box = (pred["x1"], pred["y1"], pred["x2"], pred["y2"])
        draw.rectangle(box, outline=color, width=3)
        label = (f"Y:{pred['yolo_class']} {pred['yolo_conf']:.2f} | "
                 f"D:{pred['densenet_class']} {pred['densenet_conf']:.2f}")
        draw.text((box[0], max(0, box[1] - 16)), label, fill=color)
    image.save(SAMPLE_DIR / f"{image_path.stem}_cascade.png")
print("Saved sample images:", len(list(SAMPLE_DIR.glob("*.png"))))


In [ ]:
# ============================================================
# 11. SOURCE TRACEABILITY + RESULT ZIP
# ============================================================
source_record = {
    "experiment_id": EXPERIMENT_ID,
    "yolo_experiment_id": YOLO_EXPERIMENT_ID,
    "yolo_best_pt_sha256": YOLO_BEST_SHA256,
    "densenet_experiment_id": DENSENET_EXPERIMENT_ID,
    "densenet_deployment_seed": DENSENET_DEPLOYMENT_SEED,
    "densenet_deployment_model_sha256": DENSENET_MODEL_SHA256,
    "yolo_dataset_file_id": YOLO_DATASET_FILE_ID,
    "yolo_dataset_sha256": DATASET_SHA256,
    "detection_confidence": DETECT_CONF,
    "nms_iou": NMS_IOU,
    "matching_iou": MATCH_IOU,
    "densenet_input": "raw RGB float32 0..255; bilinear 256x256; preprocessing inside model",
    "tensorflow_version": tf.__version__,
    "ultralytics_version": ultralytics.__version__,
    "python_version": sys.version.split()[0],
    "warning": "DenseNet024 was trained on whole leaves, not lesion crops; cascade metrics are exploratory.",
}
(RESULT_DIR / "source_and_config.json").write_text(
    json.dumps(source_record, indent=2, ensure_ascii=False), encoding="utf-8"
)
RESULT_ZIP = shutil.make_archive(str(WORK_ROOT / EXPERIMENT_ID), "zip", root_dir=RESULT_DIR)
with zipfile.ZipFile(RESULT_ZIP) as archive:
    bad = archive.testzip()
if bad is not None:
    raise IOError(f"Corrupt result ZIP entry: {bad}")
print("Result ZIP:", RESULT_ZIP, "SHA256:", file_digest(RESULT_ZIP))


In [ ]:
# ============================================================
# 12. UPLOAD RESULT ZIP TO DRIVE AND VERIFY
# ============================================================
archive_path = Path(RESULT_ZIP)
local_md5 = file_digest(archive_path, "md5")
existing = drive_service.files().list(
    q=f"'{YOLO_FOLDER_ID}' in parents and name = '{archive_path.name}' and trashed = false",
    fields="files(id,name,size,md5Checksum,webViewLink)", pageSize=100,
    supportsAllDrives=True, includeItemsFromAllDrives=True,
).execute().get("files", [])
same = next((item for item in existing if item.get("md5Checksum") == local_md5), None)
if same:
    uploaded = same
else:
    upload_name = archive_path.name
    if existing:
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        upload_name = f"{archive_path.stem}_{stamp}{archive_path.suffix}"
    media = MediaFileUpload(
        str(archive_path), mimetype="application/zip",
        chunksize=8 * 1024 * 1024, resumable=True,
    )
    request = drive_service.files().create(
        body={"name": upload_name, "parents": [YOLO_FOLDER_ID]}, media_body=media,
        fields="id,name,size,md5Checksum,parents,webViewLink", supportsAllDrives=True,
    )
    uploaded = None
    while uploaded is None:
        progress, uploaded = request.next_chunk(num_retries=3)
        if progress is not None:
            print(f"Upload: {progress.progress():.0%}")
verified = drive_service.files().get(
    fileId=uploaded["id"],
    fields="id,name,size,md5Checksum,parents,webViewLink", supportsAllDrives=True,
).execute()
if int(verified["size"]) != archive_path.stat().st_size:
    raise IOError("Drive result size mismatch")
if verified.get("md5Checksum") != local_md5:
    raise IOError("Drive result MD5 mismatch")
if YOLO_FOLDER_ID not in verified.get("parents", []):
    raise IOError("Drive result is not in YOLO folder")
DRIVE_RESULT_URL = verified.get("webViewLink") or f"https://drive.google.com/file/d/{verified['id']}/view"
print("RESULT UPLOADED TO DRIVE:", DRIVE_RESULT_URL)


## Cách đọc kết quả

`test_box_metrics.csv` so sánh YOLO-only với cascade ở cùng confidence 0.25 và IoU match 0.50. Với cascade, DenseNet trả `healthy` hoặc `not_cashew_leaf` thì box bị từ chối: không tính FP của class bệnh, nhưng GT không được match sẽ tính FN. Đây chỉ là kết quả trên Test V6, **không phải** bằng chứng cải thiện tổng quát hay phép kiểm định DenseNet trên crop độc lập. Cần một tập crop/box Validation riêng để chọn chính sách kết hợp trước khi đánh giá blind Test.
